In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
import torch.nn.functional as F
from tqdm import tqdm

In [2]:
device = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print(f"Using device: {device}")

Using device: mps


In [ ]:
weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1
transform = weights.transforms()

train_ds = ImageFolder(root="faces/train", transform=transform)
valid_ds = ImageFolder(root="faces/dev", transform=transform)

batch_size = 128
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4)
valid_dl = DataLoader(valid_ds, batch_size=batch_size, num_workers=4)

num_classes = len(train_ds.classes)
print(f"Number of identities: {num_classes}")

Number of identities: 7001


In [4]:
class MobileFaceEmbedding(nn.Module):
    def __init__(self, embedding_size=512, pretrained=True):
        super().__init__()
        base = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None)
        base.classifier = nn.Identity()
        self.backbone = base
        self.embedding = nn.Sequential(
            nn.Linear(576, 1024),
            nn.Hardswish(),
            nn.Linear(1024, embedding_size)
        )
        self.bn = nn.BatchNorm1d(embedding_size)

    def forward(self, x):
        x = self.backbone(x)
        x = self.embedding(x)
        x = self.bn(x)
        x = nn.functional.normalize(x, dim=1)
        return x

In [5]:
class ArcFaceLoss(nn.Module):
    def __init__(self, embedding_size, num_classes, s=30.0, m=0.5):
        super().__init__()
        self.W = nn.Parameter(torch.randn(num_classes, embedding_size))
        nn.init.xavier_uniform_(self.W)
        self.s = s
        self.m = m
        self.ce = nn.CrossEntropyLoss()

    def forward(self, embeddings, labels):
        W = nn.functional.normalize(self.W, dim=1)
        logits = torch.matmul(embeddings, W.t())
        theta = torch.acos(torch.clamp(logits, -1.0 + 1e-7, 1.0 - 1e-7))
        target_logits = torch.cos(theta + self.m)
        one_hot = torch.zeros_like(logits)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)
        output = self.s * (one_hot * target_logits + (1 - one_hot) * logits)
        loss = self.ce(output, labels)
        return loss

In [ ]:
embedding_size = 512
model = MobileFaceEmbedding(embedding_size=embedding_size, pretrained=True).to(device)
criterion = ArcFaceLoss(embedding_size=embedding_size, num_classes=num_classes, s=64.0, m=0.5).to(device)

In [7]:
def freeze_bn(model, train_affine=True):
    """Freeze all BatchNorm layers: use stored running stats, stop updates."""
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            m.eval()  # use running mean/var, stop updating them
            # freeze affine params:
            m.weight.requires_grad = train_affine
            m.bias.requires_grad = train_affine

Phase 1: Freeze backbone, train only embedding + BN

In [ ]:
for param in model.backbone.parameters():
    param.requires_grad = False
for param in model.embedding.parameters():
    param.requires_grad = True

freeze_bn(model, train_affine=False)

optimizer = torch.optim.AdamW([
    {"params": filter(lambda p: p.requires_grad, model.parameters())},
    {"params": criterion.parameters()}
], lr=3e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
epochs_phase1 = 5

for epoch in range(epochs_phase1):
    model.train()
    running_loss = 0.0

    for X, y in tqdm(train_dl, desc=f"Phase1 Epoch {epoch+1}/{epochs_phase1}"):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        embeddings = model(X)
        loss = criterion(embeddings, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()
    print(f"Phase1 Epoch [{epoch+1}/{epochs_phase1}] Loss: {running_loss/len(train_dl):.4f}")

    model.eval()
    val_loss = 0
    val_acc = 0
    with torch.no_grad():
        for Xv, yv in valid_dl:
            Xv, yv = Xv.to(device), yv.to(device)
            emb = model(Xv)
            loss_v = criterion(emb, yv)
            val_loss += loss_v.item()
    
            W = F.normalize(criterion.W, dim=1)
            logits = torch.matmul(emb, W.t())
            preds = logits.argmax(dim=1)
            val_acc += (preds == yv).float().mean().item()

    val_loss /= len(valid_dl)
    print(f"Validation Loss: {val_loss:.4f}")
    val_acc /= len(valid_dl)
    print(f"Validation Accuracy: {val_acc:.4f}")

Phase1 Epoch 1/5: 100%|██████████| 1094/1094 [02:03<00:00,  8.85it/s]

Phase1 Epoch [1/5] Loss: 39.1946


Validation Loss: 38.0769
Validation Accuracy: 0.0755


Phase1 Epoch 2/5: 100%|██████████| 1094/1094 [02:03<00:00,  8.86it/s]

Phase1 Epoch [2/5] Loss: 36.3344


...

Phase1 Epoch [4/5] Loss: 31.7646


Validation Loss: 36.1023
Validation Accuracy: 0.2633


Phase1 Epoch 5/5: 100%|██████████| 1094/1094 [04:56<00:00,  3.69it/s] 

Phase1 Epoch [5/5] Loss: 30.3633


Validation Loss: 35.9538
Validation Accuracy: 0.2759


Phase 2: Unfreeze all layers for full fine-tuning

In [ ]:
for param in model.parameters():
    param.requires_grad = True
# for name, param in model.backbone.features.named_parameters():
#     # Unfreeze only the last few inverted residual blocks
#     if "9" in name or "10" in name or "11" in name or "12" in name:
#         param.requires_grad = True

freeze_bn(model, train_affine=True)

optimizer = torch.optim.AdamW([
    {"params": filter(lambda p: p.requires_grad, model.parameters())},
    {"params": criterion.parameters()}
], lr=3e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
epochs_phase2 = 20

for epoch in range(epochs_phase2):
    model.train()
    running_loss = 0.0

    for X, y in tqdm(train_dl, desc=f"Phase2 Epoch {epoch+1}/{epochs_phase2}"):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        embeddings = model(X)
        loss = criterion(embeddings, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()
    print(f"Phase2 Epoch [{epoch+1}/{epochs_phase2}] Loss: {running_loss/len(train_dl):.4f}")

    model.eval()
    val_loss = 0
    val_acc = 0
    with torch.no_grad():
        for Xv, yv in valid_dl:
            Xv, yv = Xv.to(device), yv.to(device)
            emb = model(Xv)
            loss_v = criterion(emb, yv)
            val_loss += loss_v.item()
    
            W = F.normalize(criterion.W, dim=1)
            logits = torch.matmul(emb, W.t())
            preds = logits.argmax(dim=1)
            val_acc += (preds == yv).float().mean().item()

    val_loss /= len(valid_dl)
    print(f"Validation Loss: {val_loss:.4f}")
    val_acc /= len(valid_dl)
    print(f"Validation Accuracy: {val_acc:.4f}")

Phase2 Epoch 1/20: 100%|██████████| 1094/1094 [06:10<00:00,  2.95it/s]

Phase2 Epoch [1/20] Loss: 31.3035


Validation Loss: 32.9831
Validation Accuracy: 0.4703


Phase2 Epoch 2/20: 100%|██████████| 1094/1094 [06:12<00:00,  2.94it/s]

Phase2 Epoch [2/20] Loss: 27.4453


...

Phase2 Epoch [19/20] Loss: 11.4748


Validation Loss: 19.2183
Validation Accuracy: 0.8428


Phase2 Epoch 20/20: 100%|██████████| 1094/1094 [08:32<00:00,  2.13it/s] 

Phase2 Epoch [20/20] Loss: 11.3992


Validation Loss: 19.2087
Validation Accuracy: 0.8432


In [10]:
torch.save(model.state_dict(), "mobileface_arcface_finetuned.pth")
print("Training complete — model saved as mobileface_arcface_finetuned.pth")

Training complete — model saved as mobileface_arcface_finetuned.pth
